In [1]:
# import all libraries
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.utils.data import Dataset
import sys
from pathlib import Path

# change path to the project root
base_path = Path.cwd() / "../../"
sys.path.append(str(base_path.resolve()))

# import custom functions
from utils.classification import train_bert
from utils.evaluation import run_testset_sentiment

In [2]:
# load the data
with open("../../01_data/annotations_reduced.json", "r") as f:
    data = json.load(f)

# initialize dictionary for the sentiment classes
sent_dict = set()

# loop through all sentences
for task in data:
    if task["annotations"]:
        for annotation in task["annotations"]:
            label = annotation["tag"][3:]
            sent_dict.add(label)

# sort the tag dictionary
label_list = sorted(sent_dict)

# dictionaries that convert from id to tag and vice versa
label_to_id = {tag: i for i, tag in enumerate(label_list)}
id_to_label = {id: label for label, id in label_to_id.items()}

In [3]:
# get all data with annotations
data_with_annotations = []
for task in data:
    if task["annotations"]:
        data_with_annotations.append(task)

# split into training and test dataset
train_dataset, test_dataset = train_test_split(data_with_annotations, test_size=0.2, random_state=42)

class StanceDataset(Dataset):
    def __init__(self, data, tokenizer, label2id, max_len=128):
        self.dataset = []
        for item in data:
            sentence = item["sentence"]
            for ann in item["annotations"]:
                span_text = ann["text"]
                label = label2id[ann["tag"][3:]]
                # combine sentence and target span
                encoded = tokenizer(
                    sentence,
                    span_text,
                    truncation=True,
                    padding="max_length",
                    max_length=max_len,
                    return_tensors="pt"
                )
                self.dataset.append({
                    "input_ids": encoded["input_ids"].squeeze(0),
                    "attention_mask": encoded["attention_mask"].squeeze(0),
                    "label": torch.tensor(label, dtype=torch.long)
                })
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        return self.dataset[idx]

In [ ]:
# tune models

In [7]:
# models to train and empty dictionary to save results
model_names = ["roberta-base"]#, "bert-base-cased", "distilbert-base-cased"]
test_metrics = {}

# define global parameters
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
num_labels = len(label_to_id)

# loop
for model_name in model_names:

    # create new tokenizer depending on model
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # create new datasets and dataloaders for the specific model
    train_dataset_tensor = StanceDataset(train_dataset, tokenizer, label_to_id)
    test_dataset_tensor = StanceDataset(test_dataset, tokenizer, label_to_id)
    train_loader = DataLoader(train_dataset_tensor, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_dataset_tensor, batch_size=16, shuffle=False)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels).to(device)

    # define hyperparameters specific to the model 
    epochs = 20
    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

    # train the model and directly run it through the test set
    train_bert(train_loader, model, optimizer, epochs, device, "sentiment")
    true_labels, pred_labels = run_testset_sentiment(model=model, test_dataloader=test_loader, device=device)

    # evaluate the results and save in dictionary
    metrics = classification_report(
        [list(label_to_id.keys())[i] for i in true_labels],
        [list(label_to_id.keys())[i] for i in pred_labels],
        output_dict=True
        )
    test_metrics[model_name] = {
        "negative": metrics["neg"]["f1-score"],
        "neutral": metrics["neutral"]["f1-score"],
        "positive": metrics["pos"]["f1-score"]
    }

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/20


Training: 100%|██████████| 103/103 [00:44<00:00,  2.33it/s, loss=0.489]


Average training loss: 0.7292
Epoch 2/20


Training: 100%|██████████| 103/103 [00:43<00:00,  2.35it/s, loss=0.25] 


...

Average training loss: 0.0074
Epoch 19/20


Training: 100%|██████████| 103/103 [00:43<00:00,  2.35it/s, loss=0.000104]


Average training loss: 0.0189
Epoch 20/20


Training: 100%|██████████| 103/103 [00:43<00:00,  2.36it/s, loss=7.33e-5]


Average training loss: 0.0265


In [8]:
# export the test metrics
test_metrics

{'roberta-base': {'negative': 0.7096774193548387,
  'neutral': 0.5867768595041323,
  'positive': 0.8114901256732495}}